# ⚡ Hermes Agent x Colab T4 GPU 로컬 LLM 서버
**Qwen 3.5** 최신 오픈소스 모델을 무료 Google Colab T4 GPU에 올리고, **Hermes Agent**에 연결할 수 있는 공인 OpenAI 호환 API 엔드포인트를 생성합니다.

---
### 🚀 사용 방법
1. 상단 메뉴 `런타임` -> `런타임 유형 변경` -> **T4 GPU** 선택 확인
2. 1번부터 4번 셀까지 순서대로 재생 버튼(실행) 클릭
3. 3번 셀에 출력되는 `https://xxxx.trycloudflare.com/v1` 주소를 Hermes Agent에 연결!

In [ ]:
# [1단계] GPU 상태 확인 및 Ollama 엔진 설치 (약 10초)
!nvidia-smi
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

In [ ]:
# [2단계] Qwen 3.5 최신 모델 다운로드 (Colab T4 15GB VRAM 최적화)
# Qwen 3.5 9B (256K 컨텍스트, 네이티브 멀티모달, 에이전트/코딩 최강)
!ollama pull qwen3.5:9b

# 초경량 버전을 원할 경우 (초고속):
# !ollama pull qwen3.5:4b
# !ollama pull qwen3.5:2b

In [ ]:
# [3단계] 무료 Cloudflare 공인 터널 생성 (Hermes Agent 연결용)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import os, subprocess, time, re

# 로그 파일 방식으로 터널 실행 (블로킹 방지)
if os.path.exists('tunnel.log'):
    os.remove('tunnel.log')

subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--logfile', 'tunnel.log'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

tunnel_url = None
print('⏳ 공인 터널 주소 생성 중...')
for _ in range(30):
    time.sleep(1)
    if os.path.exists('tunnel.log'):
        with open('tunnel.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print('\n' + '='*60)
    print('🎉 [Hermes Agent 연결 정보]')
    print(f'🔹 OpenAI 호환 API Base: {tunnel_url}/v1')
    print('🔹 모델명: qwen3.5:9b')
    print('='*60)
    print('\n📌 Termux / 폰에서 Hermes Agent 실행할 때:')
    print(f'export LLM_API_BASE="{tunnel_url}/v1"')
    print('export LLM_MODEL="qwen3.5:9b"')
    print('python3 run_web.py')
else:
    print('❌ 터널 생성 확인 실패. tunnel.log를 확인해보세요.')

In [ ]:
# [4단계] 코랩 웹 챗 UI (Gradio) - 폰 브라우저에서 바로 대화 테스트!
!pip install -q gradio requests
import gradio as gr
import requests

def chat(message, history):
    payload = {
        "model": "qwen3.5:9b",
        "messages": [{"role": "user", "content": message}],
        "stream": False
    }
    try:
        resp = requests.post("http://localhost:11434/api/chat", json=payload, timeout=60)
        return resp.json()["message"]["content"]
    except Exception as e:
        return f"에러: {e}"

gr.ChatInterface(chat, title="Hermes Brain - Qwen 3.5 9B").launch(share=True)